In [ ]:
%cd ../..

import os
import torch
from tqdm import tqdm
from omegaconf import OmegaConf
from glob import glob
import numpy as np
import random
from scipy.stats import truncnorm
import pandas as pd
import SimpleITK as sitk

from dinov2.inference import generate_embeddings, build_model, view_volume, is_HU

In [ ]:
def load_mhd(path):
    image_obj = sitk.ReadImage(path)
    image = sitk.GetArrayFromImage(image_obj)
    spacing = image_obj.GetSpacing()
    spacing = np.array(spacing)[::-1]
    assert abs(spacing[2] - spacing[1]) < 0.001

    image = torch.from_numpy(image).float()
    image = image.clip(-1000, 1900)

    assert is_HU(image)

    return image, spacing

def crop_pos_mdh(path, posx, posy, posz, diameter, pad=10):
    image_obj = sitk.ReadImage(path)
    image = sitk.GetArrayFromImage(image_obj)
    image = image.clip(-1000, 1900)

    D, W, H = image.shape
    
    origin = np.array(image_obj.GetOrigin())
    spacing = np.array(image_obj.GetSpacing())
    spacing = spacing[::-1]

    radius = [3, pad, pad]

    coordx = int((posx - origin[0])/spacing[2])
    coordy = int((posy - origin[1])/spacing[1])
    coordz = int((posz - origin[2])/spacing[0])

    xmin, xmax = coordx - radius[2], coordx + radius[2]
    ymin, ymax = coordy - radius[1], coordy + radius[1]
    zmin, zmax = coordz - radius[0], coordz + radius[0]

    xmin = max(0, xmin)
    ymin = max(0, ymin)
    zmin = max(0, zmin)

    zmax = min(zmax,D)
    ymax = min(ymax, W)
    xmax = min(xmax, H)

    slice_obj = (slice(zmin, zmax, None), slice(ymin,ymax, None), slice(xmin, xmax, None))

    cropped_img = image[slice_obj]
    cropped_img = torch.from_numpy(cropped_img).float()
    
    return cropped_img, spacing


In [ ]:
dataset_root = "/data/work/vm/radio-foundation/LUNA16"
img_paths = glob(os.path.join(dataset_root,"subset*/**/*.mhd"), recursive=True)
id_to_path = {p.split("/")[-1].replace(".mhd", ""): p for p in img_paths}
len(img_paths)

In [ ]:
candidates_df = pd.read_csv(os.path.join(dataset_root, "candidates.csv"))
candidatesv2_df = pd.read_csv(os.path.join(dataset_root, "candidates_V2.csv"))
annotations_df = pd.read_csv(os.path.join(dataset_root, "annotations.csv"))
annotations_df

In [ ]:
max_rows = 1186 * 5
candidatesv2_df = candidatesv2_df[candidatesv2_df["class"] == 0]
candidatesv2_df = candidatesv2_df.sample(n=max_rows, random_state=42)

candidatesv2_df

In [ ]:
mean = annotations_df["diameter_mm"].mean()
sigma = annotations_df["diameter_mm"].std()
a, b = (mean - sigma) / sigma, (mean + sigma) / sigma
trunc_normal = truncnorm(a, b, loc=mean, scale=sigma)

def gen_diameter():
    return trunc_normal.rvs(size=1)[0]

In [ ]:
seriesuid, coordX, coordY, coordZ, diameter = annotations_df.iloc[4]
path = id_to_path[seriesuid]

In [ ]:
seriesuid, coordX, coordY, coordZ, _ = candidatesv2_df.iloc[19]
path = id_to_path[seriesuid]
diameter = gen_diameter()

In [ ]:
img, spacing = crop_pos_mdh(path, coordX, coordY, coordZ, diameter)
#img, spacing = load_mhd(path)
view_volume(img, spacing)

In [ ]:
config_path = "/home/48078029W/projects/radio-foundation/runs/base10pat/config.yaml"
checkpoint_path = "/home/48078029W/projects/radio-foundation/runs/base10pat/eval/training_99999/teacher_checkpoint.pth"

device = torch.device("cuda")

config = OmegaConf.load(config_path)
model, autocast_ctx = build_model(checkpoint_path, config, img_size=504, device=device)

data_kwargs = dict(
    fmean = -573.8,
    fstd = 461.3,
    channels = 10,
    img_size = 98,
    patch_size = 14,
    device="cuda",
    block_size=64,
    autocast_ctx=autocast_ctx,
    no_crop=True
)

In [ ]:
output_path = "/data/work/vm/radio-foundation/embeddings/LUNA16/nodules"
os.makedirs(output_path, exist_ok=True)

for idx, row in tqdm(annotations_df.iterrows()):
    seriesuid, coordX, coordY, coordZ, diameter = row
    img_path = id_to_path[seriesuid]
    
    output = {}

    img, _ = crop_pos_mdh(img_path, coordX, coordY, coordZ, diameter, pad=30)
    img = torch.nn.functional.interpolate(
        img.unsqueeze(0).unsqueeze(0), size=(data_kwargs["channels"], data_kwargs["img_size"], data_kwargs["img_size"]), mode="trilinear"
    ).squeeze(0).squeeze(0)

    collated_features = generate_embeddings(
        img,
        model=model,
        **data_kwargs # type: ignore
    )

    output[f"cls"] = collated_features["cls"]

    torch.save(output, os.path.join(output_path, f"{idx:06}.pth"))


In [ ]:
output_path = "/data/work/vm/radio-foundation/embeddings/LUNA16/negatives"
os.makedirs(output_path, exist_ok=True)

for idx, row in tqdm(candidatesv2_df.iterrows()):
    seriesuid, coordX, coordY, coordZ, _ = row
    img_path = id_to_path[seriesuid]
    
    output = {}

    diameter = gen_diameter()
    img, _ = crop_pos_mdh(img_path, coordX, coordY, coordZ, diameter, pad=30)
    img = torch.nn.functional.interpolate(
        img.unsqueeze(0).unsqueeze(0), size=(data_kwargs["channels"], data_kwargs["img_size"], data_kwargs["img_size"]), mode="trilinear"
    ).squeeze(0).squeeze(0)

    collated_features = generate_embeddings(
        img,
        model=model,
        **data_kwargs # type: ignore
    )

    output[f"cls"] = collated_features["cls"]

    torch.save(output, os.path.join(output_path, f"{idx:06}.pth"))
